# Task 3: Compare the two models and look at the errors
Mohammed made a Naive Bayes model, and Abdulrahman made a Logistic Regression model.
Both used the same train data and the same test data.
Here we compare them and we look at the messages they got wrong.

## 1. Load the predictions

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix
nb = pd.read_csv('../data/pred_nb.csv')
lr = pd.read_csv('../data/pred_lr.csv')
print(nb.shape, lr.shape)
print('Same messages in both files:', (nb['text'] == lr['text']).all())
y = (nb['label'] == 'spam').astype(int)
nb.head()

(1026, 4) (1026, 4)
Same messages in both files: True


,label,text,pred,prob_spam
0,ham,"sir, you will receive the account no another 1...",ham,0.0190
1,ham,You at mu? You should try to figure out how mu...,ham,0.0005
2,ham,The greatest test of courage on earth is to be...,ham,0.0094
3,ham,Gud mrng dear hav a nice day,ham,0.0006
4,ham,I wonder if you'll get this text?,ham,0.0171


## 2. The results side by side
We also add a very simple model that always says "ham",
to see what our models really add.

In [2]:
rows = []
for name, d in [('Naive Bayes', nb), ('Logistic Regression', lr)]:
    p = (d['pred'] == 'spam').astype(int)
    rows.append([name,
                 round(100 * accuracy_score(y, p), 1),
                 round(100 * precision_score(y, p), 1),
                 round(100 * recall_score(y, p), 1),
                 round(100 * f1_score(y, p), 1)])
rows.append(['Always say ham', round(100 * (y == 0).mean(), 1), 0.0, 0.0, 0.0])
table = pd.DataFrame(rows, columns=['model', 'accuracy', 'precision', 'recall', 'f1'])
print(table.to_string(index=False))

              model  accuracy  precision  recall   f1
        Naive Bayes      98.9      100.0    91.2 95.4
Logistic Regression      99.1       97.5    95.2 96.4
     Always say ham      87.8        0.0     0.0  0.0


## 3. The confusion matrix of each model
FP (false alarm) = a normal message sent to the spam box.
FN (missed) = a spam message that stayed in the inbox.

In [3]:
for name, d in [('Naive Bayes', nb), ('Logistic Regression', lr)]:
    cm = confusion_matrix(y, (d['pred'] == 'spam').astype(int))
    print(name)
    print(pd.DataFrame(cm, index=['true ham', 'true spam'],
                       columns=['said ham', 'said spam']))
    print('False alarms:', cm[0, 1], '| Missed spam:', cm[1, 0])
    print()

Naive Bayes
           said ham  said spam
true ham        901          0
true spam        11        114
False alarms: 0 | Missed spam: 11

Logistic Regression
           said ham  said spam
true ham        898          3
true spam         6        119
False alarms: 3 | Missed spam: 6



## 4. Where do the two models disagree?

In [4]:
both = nb.copy()
both['lr_pred'] = lr['pred']
both['lr_prob'] = lr['prob_spam']
both = both.rename(columns={'pred': 'nb_pred', 'prob_spam': 'nb_prob'})
print('They agree on', (both['nb_pred'] == both['lr_pred']).sum(), 'messages')
print('They disagree on', (both['nb_pred'] != both['lr_pred']).sum(), 'messages')
dis = both[both['nb_pred'] != both['lr_pred']]
print(dis[['label', 'nb_pred', 'lr_pred']].value_counts())

They agree on 1018 messages
They disagree on 8 messages
label  nb_pred  lr_pred
spam   ham      spam       5
ham    ham      spam       3
Name: count, dtype: int64


## 5. The spam messages we missed
We look at the spam messages that each model called ham.

In [5]:
nb_miss = both[(both['label'] == 'spam') & (both['nb_pred'] == 'ham')]
lr_miss = both[(both['label'] == 'spam') & (both['lr_pred'] == 'ham')]
same = set(nb_miss['text']) & set(lr_miss['text'])
print('Naive Bayes missed:', len(nb_miss))
print('Logistic Regression missed:', len(lr_miss))
print('Both missed the same:', len(same))
print()
for t in lr_miss['text']:
    print('-', t[:120])

Naive Bayes missed: 11
Logistic Regression missed: 6
Both missed the same: 6

- TBS/PERSOLVO. been chasing us since Sept for£38 definitely not paying now thanks to your information. We will ignore the
- Do you ever notice that when you're driving, anyone going slower than you is an idiot and everyone driving faster than y
- Would you like to see my XXX pics they are so hot they were nearly banned in the uk!
- Burger King - Wanna play footy at a top stadium? Get 2 Burger King before 1st Sept and go Large or Super with Coca-Cola 
- LIFE has never been this much fun and great until you came in. You made it truly special for me. I won't forget you! enj
- Sorry I missed your call let's talk when you have the time. I'm on 07090201529


## 6. The false alarms
These are normal messages that Logistic Regression sent to the spam box.
Naive Bayes had none.

In [6]:
lr_false = both[(both['label'] == 'ham') & (both['lr_pred'] == 'spam')]
for row in lr_false.itertuples():
    print(round(row.lr_prob, 3), '|', row.text[:130])

0.601 | Glad to see your reply.
0.992 | Hi, Mobile no.  &lt;#&gt;  has added you in their contact list on www.fullonsms.com It s a great place to send free sms to people 
0.544 | Are you free now?can i call now?


## 7. Why did they miss these messages?

In [7]:
spam = both[both['label'] == 'spam']
print('Average length of all spam   :', round(spam['text'].str.len().mean()))
print('Average length of missed spam:', round(nb_miss['text'].str.len().mean()))
print('NB probability on missed spam:', round(nb_miss['nb_prob'].max(), 3), '(highest)')
print('LR probability on missed spam:', round(lr_miss['lr_prob'].max(), 3), '(highest)')

Average length of all spam   : 140
Average length of missed spam: 132
NB probability on missed spam: 0.494 (highest)
LR probability on missed spam: 0.292 (highest)
